# Personalized Healthcare Recommendations - ML Project

## Advanced Machine Learning for Clinical Decision Support

This project develops an end-to-end machine learning system that generates personalized healthcare recommendations based on patient health data.

**Key Objectives:**
1. Predictive Modeling: Build ML models to predict personalized healthcare recommendations
2. Data-Driven Insights: Identify patterns in patient health data
3. Clinical Actionability: Generate interpretable, clinically relevant recommendations
4. Scalability: Design a system ready for real-world healthcare applications

**Technologies:** Python | Pandas | Scikit-learn | XGBoost


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

print('✓ All libraries imported successfully!')


✓ All libraries imported successfully!


In [2]:
np.random.seed(42)
n_samples = 1000

data = {
    'Age': np.random.randint(20, 80, n_samples),
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'BloodPressure_Systolic': np.random.normal(120, 15, n_samples).astype(int),
    'BloodPressure_Diastolic': np.random.normal(80, 10, n_samples).astype(int),
    'Cholesterol': np.random.normal(200, 40, n_samples).astype(int),
    'Glucose': np.random.normal(100, 25, n_samples).astype(int),
    'Hemoglobin': np.random.normal(14, 2, n_samples),
    'HeartRate': np.random.randint(60, 100, n_samples),
    'BMI': np.random.normal(26, 4, n_samples),
    'SmokingStatus': np.random.choice(['Non-smoker', 'Former-smoker', 'Current-smoker'], n_samples),
    'ExerciseLevel': np.random.choice(['Sedentary', 'Light', 'Moderate', 'Vigorous'], n_samples),
    'AlcoholConsumption': np.random.choice(['None', 'Moderate', 'Heavy'], n_samples),
    'StressLevel': np.random.choice(['Low', 'Moderate', 'High'], n_samples),
    'SleepHours': np.random.normal(7, 1.5, n_samples),
    'DiabetesHistory': np.random.choice(['No', 'Yes'], n_samples, p=[0.85, 0.15]),
    'HeartDiseaseHistory': np.random.choice(['No', 'Yes'], n_samples, p=[0.90, 0.10]),
    'Medication': np.random.choice(['None', 'Antihypertensive', 'Statin', 'Multiple'], n_samples),
}

df = pd.DataFrame(data)

# Generate recommendations based on risk score
recommendations = []
for idx, row in df.iterrows():
    risk_score = 0
    if row['BloodPressure_Systolic'] > 140 or row['BloodPressure_Diastolic'] > 90:
        risk_score += 2
    if row['Cholesterol'] > 240:
        risk_score += 2
    if row['Glucose'] > 125:
        risk_score += 2
    if row['BMI'] > 30:
        risk_score += 1
    if row['SmokingStatus'] == 'Current-smoker':
        risk_score += 2
    if row['ExerciseLevel'] == 'Sedentary':
        risk_score += 1
    if row['StressLevel'] == 'High':
        risk_score += 1
    if row['DiabetesHistory'] == 'Yes' or row['HeartDiseaseHistory'] == 'Yes':
        risk_score += 2
    
    if risk_score == 0:
        rec = 'No Action Needed'
    elif risk_score <= 3:
        rec = 'Preventive Check-up'
    elif risk_score <= 6:
        rec = 'Lifestyle Changes'
    else:
        rec = 'Medication'
    recommendations.append(rec)

df['Recommendation'] = recommendations

# Add some missing values
missing_indices = np.random.choice(df.index, size=int(0.05 * len(df)), replace=False)
for col in ['Cholesterol', 'Glucose', 'Hemoglobin', 'BMI']:
    missing_cols = np.random.choice(missing_indices, size=5)
    df.loc[missing_cols, col] = np.nan

print(f'✓ Dataset created: {df.shape[0]} patients, {df.shape[1]} features')
print(f'  Recommendation distribution:')
print(df['Recommendation'].value_counts())


✓ Dataset created: 1000 patients, 18 features
  Recommendation distribution:
Recommendation
Preventive Check-up    547
Lifestyle Changes      304
No Action Needed       108
Medication              41
Name: count, dtype: int64


In [3]:
X = df.drop('Recommendation', axis=1)
y = df['Recommendation']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Handle missing values
for col in numeric_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].mean(), inplace=True)

for col in categorical_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].mode()[0], inplace=True)

# Encode target
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Create preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f'✓ Data preprocessed: {X_train_processed.shape}')


✓ Data preprocessed: (800, 22)


In [4]:
# Feature selection
n_features = min(20, X_train_processed.shape[1])
selector = SelectKBest(f_classif, k=n_features)
X_train_selected = selector.fit_transform(X_train_processed, y_train)
X_test_selected = selector.transform(X_test_processed)

print(f'✓ Selected {n_features} best features')


✓ Selected 20 best features


In [5]:
# Train model
model = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train_selected, y_train)

# Evaluate
y_pred = model.predict(X_test_selected)
accuracy = accuracy_score(y_test, y_pred)

print(f'✓ Model trained')
print(f'  Test Accuracy: {accuracy:.4f}')


✓ Model trained
  Test Accuracy: 0.8100


In [6]:
def create_health_indices(X_data):
    X_eng = X_data.copy()
    
    if 'BloodPressure_Systolic' in X_eng.columns and 'BloodPressure_Diastolic' in X_eng.columns:
        X_eng['BP_Index'] = (X_eng['BloodPressure_Systolic'] + X_eng['BloodPressure_Diastolic']) / 2
    
    if 'Cholesterol' in X_eng.columns and 'Glucose' in X_eng.columns and 'BMI' in X_eng.columns:
        X_eng['Metabolic_Index'] = (X_eng['Cholesterol'] / 250) + (X_eng['Glucose'] / 150) + (X_eng['BMI'] / 40)
    
    return X_eng

print('✓ Feature engineering function defined')


✓ Feature engineering function defined


In [7]:
def get_recommendation(patient_data_dict):
    """
    Simple, reliable function to get healthcare recommendation for a patient.
    patient_data_dict: Dictionary with all 17 features
    """
    # Create DataFrame
    patient_df = pd.DataFrame([patient_data_dict])
    
    # Apply feature engineering
    patient_eng = create_health_indices(patient_df)
    
    # Preprocess
    patient_processed = preprocessor.transform(patient_eng)
    
    # Select features
    patient_selected = selector.transform(patient_processed)
    
    # Predict
    prediction = model.predict(patient_selected)[0]
    probabilities = model.predict_proba(patient_selected)[0]
    confidence = probabilities[prediction]
    
    # Get recommendation class
    recommendation_class = le_target.classes_[prediction]
    
    # Identify risk factors
    risk_factors = []
    if patient_data_dict['BloodPressure_Systolic'] > 140 or patient_data_dict['BloodPressure_Diastolic'] > 90:
        risk_factors.append('Elevated blood pressure')
    if patient_data_dict['Cholesterol'] > 240:
        risk_factors.append('High cholesterol')
    if patient_data_dict['Glucose'] > 125:
        risk_factors.append('Elevated glucose')
    if patient_data_dict['BMI'] > 30:
        risk_factors.append('Overweight/Obese')
    if patient_data_dict['SmokingStatus'] == 'Current-smoker':
        risk_factors.append('Current smoker')
    if patient_data_dict['ExerciseLevel'] == 'Sedentary':
        risk_factors.append('Sedentary lifestyle')
    if patient_data_dict['StressLevel'] == 'High':
        risk_factors.append('High stress')
    if patient_data_dict['DiabetesHistory'] == 'Yes':
        risk_factors.append('Diabetes history')
    if patient_data_dict['HeartDiseaseHistory'] == 'Yes':
        risk_factors.append('Heart disease history')
    
    if not risk_factors:
        risk_factors = ['None identified']
    
    return {
        'recommendation': recommendation_class,
        'confidence': float(confidence),
        'probabilities': {le_target.classes_[i]: float(probabilities[i]) for i in range(len(le_target.classes_))},
        'risk_factors': risk_factors
    }

print('✓ Recommendation function ready')


✓ Recommendation function ready


In [8]:
print('='*80)
print('PERSONALIZED HEALTHCARE RECOMMENDATIONS')
print('='*80)

# PATIENT 1: Healthy individual
patient_1 = {
    'Age': 35,
    'Gender': 'Male',
    'BloodPressure_Systolic': 120,
    'BloodPressure_Diastolic': 80,
    'Cholesterol': 190,
    'Glucose': 95,
    'Hemoglobin': 14.5,
    'HeartRate': 72,
    'BMI': 24.5,
    'SmokingStatus': 'Non-smoker',
    'ExerciseLevel': 'Vigorous',
    'AlcoholConsumption': 'Moderate',
    'StressLevel': 'Low',
    'SleepHours': 7.5,
    'DiabetesHistory': 'No',
    'HeartDiseaseHistory': 'No',
    'Medication': 'None'
}

result_1 = get_recommendation(patient_1)

print('\n🔹 PATIENT 1: Young Healthy Individual (Age 35, Male)')
print('-'*80)
print(f'📋 Profile: Non-smoker, vigorous exercise, normal vitals, no medical history')
print(f'\n✅ RECOMMENDATION: {result_1["recommendation"]}')
print(f'   Confidence: {result_1["confidence"]*100:.1f}%')
print(f'\n⚠️ Risk Factors: {", ".join(result_1["risk_factors"])}')
print(f'\n📊 Recommendation Breakdown:')
for class_name, prob in sorted(result_1['probabilities'].items(), key=lambda x: x[1], reverse=True):
    print(f'   {class_name:<30} {prob*100:5.1f}%')


PERSONALIZED HEALTHCARE RECOMMENDATIONS

🔹 PATIENT 1: Young Healthy Individual (Age 35, Male)
--------------------------------------------------------------------------------
📋 Profile: Non-smoker, vigorous exercise, normal vitals, no medical history

✅ RECOMMENDATION: No Action Needed
   Confidence: 90.6%

⚠️ Risk Factors: None identified

📊 Recommendation Breakdown:
   No Action Needed                90.6%
   Preventive Check-up              9.3%
   Lifestyle Changes                0.1%
   Medication                       0.0%


In [9]:
# PATIENT 2: High-risk individual
patient_2 = {
    'Age': 58,
    'Gender': 'Female',
    'BloodPressure_Systolic': 155,
    'BloodPressure_Diastolic': 95,
    'Cholesterol': 280,
    'Glucose': 145,
    'Hemoglobin': 12.5,
    'HeartRate': 88,
    'BMI': 32.5,
    'SmokingStatus': 'Current-smoker',
    'ExerciseLevel': 'Sedentary',
    'AlcoholConsumption': 'Heavy',
    'StressLevel': 'High',
    'SleepHours': 5.5,
    'DiabetesHistory': 'Yes',
    'HeartDiseaseHistory': 'Yes',
    'Medication': 'Multiple'
}

result_2 = get_recommendation(patient_2)

print('\n\n🔹 PATIENT 2: Middle-Aged High-Risk Individual (Age 58, Female)')
print('-'*80)
print(f'📋 Profile: Current smoker, sedentary, elevated vitals, diabetes & heart disease')
print(f'\n⚠️ RECOMMENDATION: {result_2["recommendation"]}')
print(f'   Confidence: {result_2["confidence"]*100:.1f}%')
print(f'\n🔴 Risk Factors ({len(result_2["risk_factors"])} detected):')
for factor in result_2['risk_factors']:
    print(f'   • {factor}')
print(f'\n📊 Recommendation Breakdown:')
for class_name, prob in sorted(result_2['probabilities'].items(), key=lambda x: x[1], reverse=True):
    print(f'   {class_name:<30} {prob*100:5.1f}%')

print('\n' + '='*80)
print('✅ RECOMMENDATIONS GENERATED SUCCESSFULLY')
print('='*80)




🔹 PATIENT 2: Middle-Aged High-Risk Individual (Age 58, Female)
--------------------------------------------------------------------------------
📋 Profile: Current smoker, sedentary, elevated vitals, diabetes & heart disease

⚠️ RECOMMENDATION: Medication
   Confidence: 99.2%

🔴 Risk Factors (9 detected):
   • Elevated blood pressure
   • High cholesterol
   • Elevated glucose
   • Overweight/Obese
   • Current smoker
   • Sedentary lifestyle
   • High stress
   • Diabetes history
   • Heart disease history

📊 Recommendation Breakdown:
   Medication                      99.2%
   Lifestyle Changes                0.8%
   Preventive Check-up              0.0%
   No Action Needed                 0.0%

✅ RECOMMENDATIONS GENERATED SUCCESSFULLY
